# Tomahawk Preliminary Thermal–Neutronics Screen

This notebook is a fast, reproducible Kaggle screen for the current RF-cassette concept. It combines:

- D–T neutron wall-load to neutron-flux conversion.
- Preliminary local volumetric nuclear-heating inputs for AlN, Ag–Cu–Ti braze, Mo, Mo–Cu transition material, and CuCrZr.
- RF parasitic-wall-loss uncertainty rather than incorrectly assigning the full 15 MW/module RF drive to the cassette.
- Simple coolant-flow sizing for a dedicated subcooled demineralized-water loop.
- A one-dimensional conduction/convection temperature screen and sensitivity plots.

This is not a confinement proof, a neutronics qualification, or a hardware release. The material heating values are screening inputs that must later be replaced by an OpenMC/MCNP/FENDL or equivalent transport tally using the actual CAD geometry and neutron spectrum.

## 1. Kaggle run controls

The default run is lightweight and completes quickly. Set `RUN_MODE = "smoke"` for a minimal check or `RUN_MODE = "full"` for the complete parameter sweep. No internet, package installation, uploaded datasets, or custom modules are required.

In [ ]:
RUN_MODE = "full"  # change to "smoke" for a minimal check
SAVE_OUTPUTS = True
SHOW_PLOTS = True

print(f"RUN_MODE={RUN_MODE!r}; SAVE_OUTPUTS={SAVE_OUTPUTS}; SHOW_PLOTS={SHOW_PLOTS}")

## 2. Imports and output directory

In [ ]:
import json
import math
import os
import platform
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display, Markdown
except Exception:
    display = print
    Markdown = lambda x: x

OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("tomahawk_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.figsize": (9, 5), "axes.grid": True, "grid.alpha": 0.3})

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Output directory:", OUTPUT_DIR.resolve())

## 3. Inputs

These are the current preliminary assumptions. The cell writes a copy to `tomahawk_preliminary_inputs_runtime.json` so every run has an auditable input record. The thicknesses, wall-loss fraction, heat-transfer coefficient, and volumetric heating values are intentionally editable screening parameters.

In [ ]:
INPUTS = {
    "model_name": "Tomahawk preliminary RF-cassette thermal/neutronics screen",
    "revision": "screening-0.1",
    "active_modules": 10,
    "major_radius_m": 7.5,
    "minor_radius_m": 3.0,
    "module_cad_plasma_volume_m3": 950.0,
    "neutron_wall_load_MW_m2": 4.0,
    "dt_neutron_energy_MeV": 14.1,
    "dt_total_energy_MeV": 17.6,
    "rf_power_per_module_MW": 15.0,
    "rf_wall_loss_fraction_nominal": 0.02,
    "rf_wall_loss_fraction_sweep": [0.005, 0.01, 0.02, 0.05, 0.10],
    "rf_cassette_count_per_module": 4,
    "rf_cassette_area_per_m2": 2.5,
    "plasma_heat_to_cassette_per_module_MW": 0.0,
    "coolant_name": "subcooled high-purity demineralized water",
    "coolant_cp_J_kgK": 4180.0,
    "coolant_inlet_C": 125.0,
    "coolant_pressure_MPa": 4.0,
    "coolant_deltaT_allowed_K": 30.0,
    "coolant_h_W_m2K": 20000.0,
    "cucrzr_temperature_screen_limit_C": 300.0,
    "materials": [
        {"name": "AlN insert", "density_kg_m3": 3260.0, "thickness_m": 0.0020, "conductivity_W_mK": 170.0, "qprime_n_low_W_m3": 25e6, "qprime_n_nominal_W_m3": 50e6, "qprime_n_high_W_m3": 125e6},
        {"name": "Ag-Cu-Ti braze (prototype)", "density_kg_m3": 9000.0, "thickness_m": 0.00010, "conductivity_W_mK": 40.0, "qprime_n_low_W_m3": 50e6, "qprime_n_nominal_W_m3": 100e6, "qprime_n_high_W_m3": 200e6},
        {"name": "Molybdenum face sheet", "density_kg_m3": 10220.0, "thickness_m": 0.00010, "conductivity_W_mK": 138.0, "qprime_n_low_W_m3": 75e6, "qprime_n_nominal_W_m3": 120e6, "qprime_n_high_W_m3": 250e6},
        {"name": "Mo-Cu graded transition", "density_kg_m3": 8500.0, "thickness_m": 0.0010, "conductivity_W_mK": 220.0, "qprime_n_low_W_m3": 75e6, "qprime_n_nominal_W_m3": 150e6, "qprime_n_high_W_m3": 250e6},
        {"name": "CuCrZr strap", "density_kg_m3": 8900.0, "thickness_m": 0.0050, "conductivity_W_mK": 300.0, "qprime_n_low_W_m3": 50e6, "qprime_n_nominal_W_m3": 100e6, "qprime_n_high_W_m3": 200e6},
    ],
}

# If the JSON is uploaded as a Kaggle dataset, use it automatically. The
# embedded values above remain the fallback, so the notebook runs standalone.
external_candidates = []
kaggle_input_root = Path("/kaggle/input")
if kaggle_input_root.exists():
    external_candidates = sorted(kaggle_input_root.rglob("tomahawk_preliminary_inputs.json"))
if external_candidates:
    external_path = external_candidates[0]
    INPUTS = json.loads(external_path.read_text(encoding="utf-8"))
    print("Loaded external inputs:", external_path)

if RUN_MODE == "smoke":
    INPUTS["rf_wall_loss_fraction_sweep"] = [0.02]

runtime_input_path = OUTPUT_DIR / "tomahawk_preliminary_inputs_runtime.json"
runtime_input_path.write_text(json.dumps(INPUTS, indent=2) + "\n", encoding="utf-8")
print("Saved runtime inputs:", runtime_input_path)

## 4. Input validation

In [ ]:
def validate_inputs(cfg):
    positive_keys = [
        "active_modules", "major_radius_m", "minor_radius_m", "module_cad_plasma_volume_m3",
        "neutron_wall_load_MW_m2", "dt_neutron_energy_MeV", "dt_total_energy_MeV",
        "rf_power_per_module_MW", "rf_cassette_count_per_module", "rf_cassette_area_per_m2",
        "coolant_cp_J_kgK", "coolant_deltaT_allowed_K", "coolant_h_W_m2K",
    ]
    for key in positive_keys:
        assert float(cfg[key]) > 0.0, f"{key} must be positive"
    assert 0.0 <= float(cfg["rf_wall_loss_fraction_nominal"]) <= 1.0
    assert all(0.0 <= float(x) <= 1.0 for x in cfg["rf_wall_loss_fraction_sweep"])
    assert cfg["dt_total_energy_MeV"] > cfg["dt_neutron_energy_MeV"]
    assert len(cfg["materials"]) >= 2
    for material in cfg["materials"]:
        for key in ["density_kg_m3", "thickness_m", "conductivity_W_mK", "qprime_n_low_W_m3", "qprime_n_nominal_W_m3", "qprime_n_high_W_m3"]:
            assert float(material[key]) > 0.0, f"{material['name']}: {key} must be positive"
        assert material["qprime_n_low_W_m3"] <= material["qprime_n_nominal_W_m3"] <= material["qprime_n_high_W_m3"]

validate_inputs(INPUTS)
print("Input validation: PASS")

## 5. Geometry, D–T neutron flux, and power consistency

In [ ]:
E_CHARGE_C = 1.602176634e-19
E_N_J = INPUTS["dt_neutron_energy_MeV"] * 1e6 * E_CHARGE_C
E_DT_TOTAL_J = INPUTS["dt_total_energy_MeV"] * 1e6 * E_CHARGE_C
q_n_W_m2 = INPUTS["neutron_wall_load_MW_m2"] * 1e6

major_radius = INPUTS["major_radius_m"]
minor_radius = INPUTS["minor_radius_m"]
module_torus_surface_area_m2 = 4.0 * math.pi**2 * major_radius * minor_radius
neutron_flux_n_m2_s = q_n_W_m2 / E_N_J
neutron_flux_n_cm2_s = neutron_flux_n_m2_s / 1e4
neutron_power_module_W = q_n_W_m2 * module_torus_surface_area_m2
neutron_power_plant_W = neutron_power_module_W * INPUTS["active_modules"]
fusion_power_equivalent_W = neutron_power_plant_W / (INPUTS["dt_neutron_energy_MeV"] / INPUTS["dt_total_energy_MeV"])

derived = {
    "module_torus_surface_area_m2": module_torus_surface_area_m2,
    "neutron_flux_n_m2_s": neutron_flux_n_m2_s,
    "neutron_flux_n_cm2_s": neutron_flux_n_cm2_s,
    "neutron_power_module_W": neutron_power_module_W,
    "neutron_power_plant_W": neutron_power_plant_W,
    "fusion_power_equivalent_W": fusion_power_equivalent_W,
}

print(f"Toroidal surface area per module: {module_torus_surface_area_m2:,.1f} m²")
print(f"14.1 MeV neutron flux: {neutron_flux_n_cm2_s:.3e} n/cm²/s")
print(f"Neutron power implied per module: {neutron_power_module_W/1e9:.3f} GW")
print(f"Neutron power implied for {INPUTS['active_modules']} modules: {neutron_power_plant_W/1e9:.3f} GW")
print(f"D-T fusion-power equivalent: {fusion_power_equivalent_W/1e9:.3f} GW")
print("Assumption check: the 4 MW/m² wall load is interpreted per active module.")

### Interpretation note

The 4 MW/m² value is treated as a per-module neutron wall load. If it was intended to be plant-wide, all flux and heating results must be rescaled and the 42.5 GWth plant specification is not power-consistent with the stated torus area.

## 6. Material-layer nuclear-heating screen

In [ ]:
materials_df = pd.DataFrame(INPUTS["materials"])
materials_df["qprime_n_low_MW_m3"] = materials_df["qprime_n_low_W_m3"] / 1e6
materials_df["qprime_n_nominal_MW_m3"] = materials_df["qprime_n_nominal_W_m3"] / 1e6
materials_df["qprime_n_high_MW_m3"] = materials_df["qprime_n_high_W_m3"] / 1e6
materials_df["nuclear_areal_low_W_m2"] = materials_df["qprime_n_low_W_m3"] * materials_df["thickness_m"]
materials_df["nuclear_areal_nominal_W_m2"] = materials_df["qprime_n_nominal_W_m3"] * materials_df["thickness_m"]
materials_df["nuclear_areal_high_W_m2"] = materials_df["qprime_n_high_W_m3"] * materials_df["thickness_m"]
materials_df["implied_kerma_nominal_1e-14_Gy_m2"] = (
    materials_df["qprime_n_nominal_W_m3"] / (neutron_flux_n_m2_s * materials_df["density_kg_m3"]) / 1e-14
)

display(materials_df[[
    "name", "thickness_m", "qprime_n_low_MW_m3", "qprime_n_nominal_MW_m3", "qprime_n_high_MW_m3",
    "nuclear_areal_nominal_W_m2", "implied_kerma_nominal_1e-14_Gy_m2"
]])

q_nuclear_areal_low_W_m2 = materials_df["nuclear_areal_low_W_m2"].sum()
q_nuclear_areal_nominal_W_m2 = materials_df["nuclear_areal_nominal_W_m2"].sum()
q_nuclear_areal_high_W_m2 = materials_df["nuclear_areal_high_W_m2"].sum()

print(f"Nominal cassette-layer neutron heating: {q_nuclear_areal_nominal_W_m2/1e6:.3f} MW/m² of cassette face area")
print("The implied kerma column is only an audit aid; replace the volumetric-heating inputs with transport tallies later.")

## 7. Integrated RF-cassette load and coolant sizing

In [ ]:
cassette_area_m2 = INPUTS["rf_cassette_count_per_module"] * INPUTS["rf_cassette_area_per_m2"]
rf_power_module_W = INPUTS["rf_power_per_module_MW"] * 1e6
plasma_heat_module_W = INPUTS["plasma_heat_to_cassette_per_module_MW"] * 1e6
cp_water = INPUTS["coolant_cp_J_kgK"]
deltaT_allowed = INPUTS["coolant_deltaT_allowed_K"]
coolant_inlet_C = INPUTS["coolant_inlet_C"]
h_coolant = INPUTS["coolant_h_W_m2K"]
cucrzr_limit_C = INPUTS["cucrzr_temperature_screen_limit_C"]

rf_loss_fractions = np.array(INPUTS["rf_wall_loss_fraction_sweep"], dtype=float)
rf_df = pd.DataFrame({"rf_wall_loss_fraction": rf_loss_fractions})
rf_df["rf_wall_loss_MW_module"] = rf_df["rf_wall_loss_fraction"] * rf_power_module_W / 1e6
rf_df["neutron_heat_MW_module_nominal"] = q_nuclear_areal_nominal_W_m2 * cassette_area_m2 / 1e6
rf_df["plasma_heat_MW_module"] = plasma_heat_module_W / 1e6
rf_df["total_cassette_heat_MW_module"] = (
    rf_df["rf_wall_loss_MW_module"] + rf_df["neutron_heat_MW_module_nominal"] + rf_df["plasma_heat_MW_module"]
)
rf_df["coolant_mass_flow_kg_s_module"] = rf_df["total_cassette_heat_MW_module"] * 1e6 / (cp_water * deltaT_allowed)
rf_df["coolant_mass_flow_kg_s_plant"] = rf_df["coolant_mass_flow_kg_s_module"] * INPUTS["active_modules"]

# Preliminary 1-D thermal resistance screen. This is not a CFD result.
R_cond_m2K_W = float((materials_df["thickness_m"] / materials_df["conductivity_W_mK"]).sum())
rf_df["rf_wall_heat_flux_MW_m2"] = rf_df["rf_wall_loss_MW_module"] / cassette_area_m2
rf_df["total_layer_heat_flux_MW_m2"] = rf_df["rf_wall_heat_flux_MW_m2"] + q_nuclear_areal_nominal_W_m2 / 1e6
rf_df["deltaT_conduction_K"] = rf_df["total_layer_heat_flux_MW_m2"] * 1e6 * R_cond_m2K_W
rf_df["deltaT_convection_K"] = rf_df["total_layer_heat_flux_MW_m2"] * 1e6 / h_coolant
rf_df["estimated_hot_surface_C"] = coolant_inlet_C + rf_df["deltaT_conduction_K"] + rf_df["deltaT_convection_K"]
rf_df["CuCrZr_screen_margin_C"] = cucrzr_limit_C - rf_df["estimated_hot_surface_C"]
rf_df["thermal_screen_pass"] = rf_df["CuCrZr_screen_margin_C"] > 0.0

display(rf_df.round(3))
print(f"Total cassette area per module: {cassette_area_m2:.2f} m²")
print(f"Layer conduction resistance: {R_cond_m2K_W:.3e} m² K/W")
print(f"Nominal coolant flow at {INPUTS['rf_wall_loss_fraction_nominal']*100:.1f}% RF wall loss: "
      f"{(rf_df.iloc[(rf_df['rf_wall_loss_fraction']-INPUTS['rf_wall_loss_fraction_nominal']).abs().argmin()]['coolant_mass_flow_kg_s_module']):.2f} kg/s/module")

## 8. Sensitivity sweep: wall load and RF wall-loss fraction

In [ ]:
wall_loads_MW_m2 = np.array([2.0, 4.0, 6.0, 8.0]) if RUN_MODE == "full" else np.array([4.0])
loss_sweep = np.array([0.005, 0.01, 0.02, 0.05, 0.10]) if RUN_MODE == "full" else np.array([0.02])
rows = []

for wall_load in wall_loads_MW_m2:
    scale = wall_load / INPUTS["neutron_wall_load_MW_m2"]
    q_areal = q_nuclear_areal_nominal_W_m2 * scale
    for loss_fraction in loss_sweep:
        rf_wall_W = loss_fraction * rf_power_module_W
        total_heat_W = rf_wall_W + q_areal * cassette_area_m2 + plasma_heat_module_W
        total_flux_W_m2 = q_areal + rf_wall_W / cassette_area_m2
        dT_cond = total_flux_W_m2 * R_cond_m2K_W
        dT_conv = total_flux_W_m2 / h_coolant
        hot_surface_C = coolant_inlet_C + dT_cond + dT_conv
        rows.append({
            "wall_load_MW_m2": wall_load,
            "rf_wall_loss_fraction": loss_fraction,
            "total_heat_MW_module": total_heat_W / 1e6,
            "coolant_flow_kg_s_module": total_heat_W / (cp_water * deltaT_allowed),
            "hot_surface_C": hot_surface_C,
            "screen_margin_C": cucrzr_limit_C - hot_surface_C,
            "screen_pass": hot_surface_C < cucrzr_limit_C,
        })

sensitivity_df = pd.DataFrame(rows)
display(sensitivity_df.round(3))
print("Sensitivity cases:", len(sensitivity_df))

## 9. Plots

In [ ]:
if SHOW_PLOTS:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].bar(materials_df["name"], materials_df["qprime_n_nominal_MW_m3"], color="#177e89")
    axes[0].set_ylabel("Nominal neutron heating (MW/m³)")
    axes[0].set_title("Material volumetric heating screen")
    axes[0].tick_params(axis="x", rotation=70)

    axes[1].plot(rf_df["rf_wall_loss_fraction"] * 100.0, rf_df["coolant_mass_flow_kg_s_module"], marker="o", label="Per module")
    axes[1].plot(rf_df["rf_wall_loss_fraction"] * 100.0, rf_df["coolant_mass_flow_kg_s_plant"], marker="s", label="10-module plant")
    axes[1].set_xlabel("RF wall-loss fraction (%)")
    axes[1].set_ylabel("Water flow (kg/s)")
    axes[1].set_title("Coolant-flow sensitivity")
    axes[1].legend()

    axes[2].plot(rf_df["rf_wall_loss_fraction"] * 100.0, rf_df["estimated_hot_surface_C"], marker="o", color="#d1495b")
    axes[2].axhline(cucrzr_limit_C, color="black", linestyle="--", label="CuCrZr screen limit")
    axes[2].set_xlabel("RF wall-loss fraction (%)")
    axes[2].set_ylabel("Estimated hot-surface temperature (°C)")
    axes[2].set_title("1-D thermal screen")
    axes[2].legend()
    plt.tight_layout()
    plot_path = OUTPUT_DIR / "tomahawk_preliminary_screen.png"
    fig.savefig(plot_path, dpi=160, bbox_inches="tight")
    plt.show()
    print("Saved plot:", plot_path)

## 10. Export results

In [ ]:
summary = {
    "model_name": INPUTS["model_name"],
    "revision": INPUTS["revision"],
    "run_mode": RUN_MODE,
    "module_torus_surface_area_m2": module_torus_surface_area_m2,
    "neutron_flux_n_cm2_s": neutron_flux_n_cm2_s,
    "nominal_cassette_nuclear_heat_MW_module": q_nuclear_areal_nominal_W_m2 * cassette_area_m2 / 1e6,
    "nominal_rf_wall_loss_fraction": INPUTS["rf_wall_loss_fraction_nominal"],
    "cucrzr_temperature_screen_limit_C": cucrzr_limit_C,
    "minimum_margin_in_sweep_C": float(rf_df["CuCrZr_screen_margin_C"].min()),
    "maximum_module_coolant_flow_kg_s": float(rf_df["coolant_mass_flow_kg_s_module"].max()),
}

summary_path = OUTPUT_DIR / "tomahawk_preliminary_summary.json"
materials_path = OUTPUT_DIR / "tomahawk_material_heating_screen.csv"
coolant_path = OUTPUT_DIR / "tomahawk_coolant_sweep.csv"
sensitivity_path = OUTPUT_DIR / "tomahawk_sensitivity_sweep.csv"

summary_path.write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
materials_df.to_csv(materials_path, index=False)
rf_df.to_csv(coolant_path, index=False)
sensitivity_df.to_csv(sensitivity_path, index=False)

print("Exported:")
for path in [runtime_input_path, summary_path, materials_path, coolant_path, sensitivity_path]:
    print(" -", path)
print("\nSummary:")
print(json.dumps(summary, indent=2))

## 11. What this run does and does not establish

The run establishes an auditable preliminary thermal/neutronics bookkeeping baseline and exposes which assumptions drive coolant flow and temperature. It does **not** establish plasma confinement, MHD stability, alpha self-heating, RF-wave coupling, structural integrity, tritium breeding, or licensing readiness.

Before hardware-specific conclusions, replace the screening volumetric-heating values with transport tallies from the actual 3-D cassette and run coupled electromagnetic, neutronics, CFD, and thermo-mechanical analyses. Preserve the exported JSON/CSV files as the run record.